
# Урок 7. Telco Churn — Моделирование и интерпретация (01)

Вводный курс **«ML для аналитиков»** — итоговый проект (часть 2 из 2).

**В этом ноутбуке:**
1) Загружаем `telco_churn_prepared.csv` (из предыдущего ноутбука).
2) Делим на train/test.
3) Обучаем 2 простые модели (Logistic Regression и RandomForest).
4) Смотрим метрики качества (accuracy, precision, recall, ROC-AUC).
5) Интерпретируем: важности признаков, SHAP (опционально).


In [ ]:
# Импорты
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, RocCurveDisplay
)

print("Импорты готовы")


## 1. Загрузка подготовленного датасета

Файл `telco_churn_prepared.csv` должен быть сохранён предыдущим ноутбуком.


In [ ]:
import os, pandas as pd

path = "telco_churn_prepared.csv"
if not os.path.exists(path):
    raise FileNotFoundError("Не найден 'telco_churn_prepared.csv'. Запустите сначала ноутбук 00 (Prep & EDA).")

df = pd.read_csv(path)
print("Загружено:", df.shape)
df.head(3)


## 2. Train/Test split и масштабирование (для логистической регрессии)


In [ ]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler(with_mean=False)  # with_mean=False: только масштабируем (без центрирования); для плотных данных подошёл бы и StandardScaler() по умолчанию
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train.shape, X_test.shape


## 3. Базовые модели
- Logistic Regression (интерпретируемость)
- RandomForest (не-линейности, важности признаков)


In [ ]:
# Логистическая регрессия
logreg = LogisticRegression(max_iter=1000, n_jobs=None)
logreg.fit(X_train_scaled, y_train)

y_pred_lr = logreg.predict(X_test_scaled)
y_proba_lr = logreg.predict_proba(X_test_scaled)[:,1]

In [ ]:
# Случайный лес
rf = RandomForestClassifier(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:,1]


## 4. Метрики качества


In [ ]:
def report_metrics(y_true, y_pred, y_proba):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    roc = roc_auc_score(y_true, y_proba)
    print(f"Accuracy: {acc:.3f} | Precision: {prec:.3f} | Recall: {rec:.3f} | F1: {f1:.3f} | ROC-AUC: {roc:.3f}")

print("Logistic Regression:")
report_metrics(y_test, y_pred_lr, y_proba_lr)

print("\nRandom Forest:")
report_metrics(y_test, y_pred_rf, y_proba_rf)

In [ ]:
# Матрица ошибок для RF
cm = confusion_matrix(y_test, y_pred_rf)
print("Confusion Matrix (RF):\n", cm)

In [ ]:

# ROC-кривые обеих моделей
fig, ax = plt.subplots(figsize=(6,4))
RocCurveDisplay.from_predictions(y_test, y_proba_lr, name="LogReg", ax=ax)
RocCurveDisplay.from_predictions(y_test, y_proba_rf, name="RandomForest", ax=ax)
ax.set_title("ROC-кривые")
plt.show()


## 5. Интерпретация

### 5.2. Важности признаков (RandomForest)

> Важности из случайного леса (impurity-based) склонны завышать непрерывные признаки (`TotalCharges`, `MonthlyCharges`, `tenure`) и признаки с большим числом значений. Это ориентир для обсуждения, а не оценка причинного влияния.


### 5.1. Коэффициенты логистической регрессии

Признаки были стандартизированы, поэтому коэффициенты сопоставимы по величине. Положительный коэффициент повышает вероятность ухода, отрицательный — снижает. Осторожно: связанные признаки (например, `tenure`, `MonthlyCharges` и `TotalCharges`, а также `MonthlyCharges` и подключённые услуги) делят вклад между собой, поэтому знак и величина отдельного коэффициента могут быть неинтуитивными.


In [ ]:
coefs = pd.Series(logreg.coef_[0], index=X.columns).sort_values()

print("Сильнее всего снижают вероятность ухода:")
display(coefs.head(8))
print("Сильнее всего повышают вероятность ухода:")
display(coefs.tail(8))


In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)
print("Топ-15 признаков по важности (RF):")
display(importances)

In [ ]:
# Простая визуализация важностей (топ-15)
plt.figure(figsize=(6,4))
importances.head(15).iloc[::-1].plot(kind="barh")
plt.title("Feature Importances (RF) — Top 15")
plt.tight_layout()
plt.show()


### 5.3. SHAP (опционально)

Может потребоваться установка:
```
!pip -q install shap
```

Расчёт на всём тестовом наборе может занять несколько минут.

In [ ]:
# Опционально: запустить SHAP в Colab (раскомментируйте две строки ниже!)
# !pip -q install shap
# import shap
# explainer = shap.TreeExplainer(rf)
# shap_values = explainer.shap_values(X_test)
# sv = shap_values[1] if isinstance(shap_values, list) else shap_values[:, :, 1]  # класс 1 = уход; формат ответа зависит от версии shap
# shap.summary_plot(sv, X_test, show=False)
# plt.title("SHAP Summary (RF)")
# plt.show()


## 6. Что обсуждаем
- Чем отличаются выводы логистической регрессии и случайного леса?
- Какие метрики критичны для churn-задачи (часто важен Recall для "уходящих")?
- Какие признаки оказались топовыми — логично ли это бизнесу?
- Как использовать прогнозы на практике: приоритизация удерживающих кампаний, персонализация офферов.
